In [0]:
# ============================================
# STEP 1 — Install Prophet
# ============================================
%pip install prophet

In [0]:
# ============================================
# STEP 2 — Imports
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet

import warnings
warnings.filterwarnings("ignore")

In [0]:
# ============================================
# STEP 3 — Load Data & Prepare for Prophet
# ============================================

daily=spark.table("workspace.default.silver_weather_daily").toPandas()
daily["date"]= pd.to_datetime(daily['date'])
daily['ds']=daily['date']
daily['y']=daily['avg_temp'] 

prophet_df = daily[['ds','y']].copy()
prophet_df.rename(columns = {"ds":"ds","y":"y"}, inplace=True)

print(f"Rows: {len(prophet_df)}")
print(f"Date range: {prophet_df['ds'].min().date()} → {prophet_df['ds'].max().date()}")
print(f"\nFirst 5 rows:")
print(prophet_df.head())
print(f"\nTarget variable (y) stats:")
print(prophet_df["y"].describe().round(2))

In [0]:
print(daily["ds"].dtype)

In [0]:
# ============================================
# STEP 4 — Visualize Raw Data (4 plots)
# ============================================
plt.close("all")
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Luxembourg Daily Temperature — What Prophet will learn from",
             fontsize=13, fontweight="bold")

# Plot 1: Full time series
axes[0,0].plot(prophet_df["ds"], prophet_df["y"],
               color="#185FA5", lw=0.8, alpha=0.7)
axes[0,0].set_title("Daily average temperature 2021–2026")
axes[0,0].set_ylabel("°C")
axes[0,0].grid(alpha=0.3)

# Plot 2: Monthly average — yearly seasonality
prophet_df["month"] = prophet_df["ds"].dt.month
monthly = prophet_df.groupby("month")["y"].mean()
axes[0,1].bar(monthly.index, monthly.values, color="#185FA5")
axes[0,1].set_title("Average temperature by month")
axes[0,1].set_ylabel("°C")
axes[0,1].set_xticks(range(1,13))
axes[0,1].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                            "Jul","Aug","Sep","Oct","Nov","Dec"])
axes[0,1].grid(axis="y", alpha=0.3)

# Plot 3: Year over year comparison
prophet_df["year"] = prophet_df["ds"].dt.year
prophet_df["dayofyear"] = prophet_df["ds"].dt.dayofyear
for year in prophet_df["year"].unique():
    subset = prophet_df[prophet_df["year"] == year]
    axes[1,0].plot(subset["dayofyear"], subset["y"],
                   lw=0.8, alpha=0.7, label=str(year))
axes[1,0].set_title("Year over year temperature comparison")
axes[1,0].set_ylabel("°C")
axes[1,0].set_xlabel("Day of year")
axes[1,0].legend(fontsize=8)
axes[1,0].grid(alpha=0.3)

# Plot 4: 30-day rolling average — trend
prophet_df["rolling"] = prophet_df["y"].rolling(30, center=True).mean()
axes[1,1].plot(prophet_df["ds"], prophet_df["y"],
               color="#185FA5", lw=0.5, alpha=0.3, label="Daily")
axes[1,1].plot(prophet_df["ds"], prophet_df["rolling"],
               color="#D85A30", lw=1.5, label="30-day rolling mean")
axes[1,1].set_title("30-day rolling average — trend over 5 years")
axes[1,1].set_ylabel("°C")
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)

plt.tight_layout()

### Analyze what Prophet will learn from these:
**Plot 1** — Daily temperature 2021–2026:

Clear repeating waves — winter low, summer high every year
Prophet sees this as yearly seasonality

**Plot 2** — Average by month:

January/February coldest (~ 2–4°C)
July/August warmest (~ 18°C)
This is the seasonality pattern Prophet will replicate in the forecast

**Plot 3** — Year over year:

All years follow almost the same curve shape
2021 starts in May (that's why it starts at day 150)
Some years slightly warmer or cooler — Prophet detects these as changepoints

**Plot 4** — 30-day rolling trend:

The red line shows the smooth underlying pattern
No strong upward or downward trend over 5 years — temperature is stable
This tells Prophet the trend component will be nearly flat.


Blue lines (thin, faded) — the actual raw daily temperature values. Every single day's temperature is plotted. They look "noisy" and jagged because daily temperatures jump up and down a lot — a warm day followed by a cold day etc.
Red line (thick) — the 30-day rolling average. It takes the average of the last 30 days and draws a smooth curve. This removes the daily noise and shows the underlying trend.
Why we show both:

**Blue** = what actually happened day by day
**Red** = the smooth pattern underneath — what Prophet will try to learn

Think of it like this:

**Blue** = reality (messy)
**Red** = the story behind reality (clean)

Prophet learns from the pattern (red) not the daily noise (blue). That is why it produces smooth forecast curves with confidence bands rather than jagged predictions.

In [0]:
# ============================================
# STEP 5 — Train / Test Split
# ============================================
plt.close("all")
# Split by time — never randomly for time series
train = prophet_df[prophet_df["ds"] < "2025-01-01"].copy()
test  = prophet_df[prophet_df["ds"] >= "2025-01-01"].copy()

print(f"Train: {len(train)} days | {train['ds'].min().date()} → {train['ds'].max().date()}")
print(f"Test:  {len(test)} days  | {test['ds'].min().date()} → {test['ds'].max().date()}")

# Visualize the split

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train["ds"], train["y"], color="#185FA5", lw=0.8, label="Train")
ax.plot(test["ds"],  test["y"],  color="#D85A30", lw=0.8, label="Test")
ax.axvline(pd.to_datetime("2025-01-01"), color="black", linestyle="--", lw=1.5, label="Split point")
ax.set_title("Train / Test split")
ax.set_ylabel("°C")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
display(fig)
plt.close(fig)

In [0]:
# ============================================
# STEP 6 — Visualize AutoML Forecast
# ============================================
plt.close("all")

forecast = spark.table("forecast_predictions_1780046331583").toPandas()
forecast["date"] = pd.to_datetime(forecast["date"])
forecast = forecast.sort_values("date")

fig, ax = plt.subplots(figsize=(16, 6))

# Confidence band
ax.fill_between(forecast["date"],
                forecast["predicted_avg_temp_lower"],
                forecast["predicted_avg_temp_upper"],
                alpha=0.2, color="#185FA5", label="Confidence interval (80%)")

# Forecast line
ax.plot(forecast["date"], forecast["predicted_avg_temp"],
        color="#185FA5", lw=2, label="Forecast (avg_temp)")

ax.set_title("SudEnergy Luxembourg — 1 Year Temperature Forecast (AutoML Prophet)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("°C")
ax.set_xlabel("Date")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
display(fig)
plt.close(fig)

In [0]:
# ============================================
# STEP 7 — Historical + Forecast Combined Chart
# ============================================
plt.close("all")

# Load historical silver data
historical = spark.table("workspace.default.silver_weather_daily").toPandas()
historical["date"] = pd.to_datetime(historical["date"])
historical = historical.sort_values("date")

# Load forecast
forecast = spark.table("forecast_predictions_1780046331583").toPandas()
forecast["date"] = pd.to_datetime(forecast["date"])
forecast = forecast.sort_values("date")

fig, ax = plt.subplots(figsize=(18, 7))

# Historical data
ax.plot(historical["date"], historical["avg_temp"],
        color="#185FA5", lw=0.8, alpha=0.5, label="Historical temperature")

# 30-day rolling average on historical
historical["rolling"] = historical["avg_temp"].rolling(30, center=True).mean()
ax.plot(historical["date"], historical["rolling"],
        color="#0D3B6E", lw=2, label="Historical 30-day average")

# Vertical line separating past from future
ax.axvline(pd.to_datetime("2026-05-29"), color="black",
           linestyle="--", lw=1.5, label="Today (forecast start)")

# Confidence band
ax.fill_between(forecast["date"],
                forecast["predicted_avg_temp_lower"],
                forecast["predicted_avg_temp_upper"],
                alpha=0.2, color="#D85A30", label="Forecast confidence (80%)")

# Forecast line
ax.plot(forecast["date"], forecast["predicted_avg_temp"],
        color="#D85A30", lw=2.5, label="Forecast avg_temp")

ax.set_title("SudEnergy Luxembourg — Historical Weather + 1 Year Forecast (AutoML Prophet)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("°C")
ax.set_xlabel("Date")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
display(fig)
plt.close(fig)

Blue thin line — 5 years of actual daily temperatures
Dark blue thick line — smooth 30-day rolling average showing the seasonal pattern clearly
Dashed black line — today, where history ends and forecast begins
Orange line — Prophet's 1-year forecast
Orange band — 80% confidence interval

## Key observations:

The forecast perfectly continues the seasonal pattern learned from 5 years of history ✓
Winter 2026–2027 forecast (~ 3–5°C) matches previous winters ✓
Summer 2026 forecast (~ 17–18°C) matches previous summers ✓
The confidence band widens in winter — correctly showing more uncertainty ✓

## This chart alone tells a powerful story to SudEnergy:

We have 5 years of clean historical data
Databricks AutoML selected Prophet as the best model automatically
The model learned Luxembourg's seasonal patterns and forecasts them forward